# Merging Annovar Outputs (genome and exome) for Gnomad Annotations

In [ ]:
import pandas as pd
from pathlib import Path
import shutil

# Make preview files

In [ ]:
csv1 = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/06_annovar_output/fixed_annovar_output_expanded_otherinfo.csv")
csv2 = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/06_annovar_output/fixed_annovar_output_with_gnomad41_genome_and_exome.csv")

In [ ]:
df1 = pd.read_csv(csv1, nrows=50, low_memory=False)
df2 = pd.read_csv(csv2, nrows=50, low_memory=False)

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
preview = pd.read_csv(csv1, nrows=50, low_memory=False)
preview_out = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/06_annovar_output/fixed_annovar_output_expanded_otherinfo_PREVIEW.csv")
preview.to_csv(preview_out, index=False)


In [ ]:
# Input files
genome_csv = Path("/project/knathans_shared/donetski/Run2_gnomad41_reannotation_with_header/genome_result/annotated_output.hg38_multianno.csv")
exome_csv = Path("/project/knathans_shared/donetski/Run2_gnomad41_reannotation_with_header/exome_result/annotated_output.hg38_multianno.csv")

# Output preview directory
preview_dir = Path("/project/knathans_shared/donetski/Run2_gnomad41_reannotation_with_header/preview_files")
preview_dir.mkdir(parents=True, exist_ok=True)

# Load only first 50 rows so the files are easier to inspect
genome_preview = pd.read_csv(genome_csv, nrows=50, low_memory=False)
exome_preview = pd.read_csv(exome_csv, nrows=50, low_memory=False)

print("Genome preview shape:", genome_preview.shape)
print("Exome preview shape:", exome_preview.shape)

# Save smaller preview files
genome_preview_out = preview_dir / "genome_annotated_output_first_50_rows.csv"
exome_preview_out = preview_dir / "exome_annotated_output_first_50_rows.csv"

genome_preview.to_csv(genome_preview_out, index=False)
exome_preview.to_csv(exome_preview_out, index=False)

print("Saved genome preview:", genome_preview_out)
print("Saved exome preview:", exome_preview_out)

# Quick column check
print("\nGenome columns:")
print(genome_preview.columns.tolist())

print("\nExome columns:")
print(exome_preview.columns.tolist())

In [ ]:
genome_csv = Path("/project/knathans_shared/donetski/Run2_gnomad41_reannotation_with_header/genome_result/annotated_output.hg38_multianno.csv")
exome_csv = Path("/project/knathans_shared/donetski/Run2_gnomad41_reannotation_with_header/exome_result/annotated_output.hg38_multianno.csv")

output_dir = Path("/project/knathans_shared/donetski/Run2_gnomad41_reannotation_with_header/fixed_combined_output")
output_dir.mkdir(parents=True, exist_ok=True)

fixed_output_csv = output_dir / "fixed_annovar_output_with_gnomad41_genome_and_exome.csv"

# Copy the genome file because it already contains both genome + exome gnomAD columns
shutil.copyfile(genome_csv, fixed_output_csv)

print("Saved fixed output:", fixed_output_csv)

# Check columns before combining

In [ ]:
genome_cols = pd.read_csv(genome_csv, nrows=0).columns.tolist()
exome_cols = pd.read_csv(exome_csv, nrows=0).columns.tolist()

genome_gnomad_cols = [c for c in genome_cols if c.startswith("gnomad41_genome")]
exome_gnomad_cols = [c for c in exome_cols if c.startswith("gnomad41_exome")]

missing_exome_cols = [c for c in exome_gnomad_cols if c not in genome_cols]

print("Genome gnomAD columns in genome file:", len(genome_gnomad_cols))
print("Exome gnomAD columns in exome file:", len(exome_gnomad_cols))
print("Exome columns missing from genome file:", len(missing_exome_cols))

print("Missing exome columns:")
print(missing_exome_cols)

In [ ]:
annovar_csv = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/06_annovar_output/fixed_annovar_output_with_gnomad41_genome_and_exome.csv")
    
df = pd.read_csv(annovar_csv, encoding="latin1", low_memory=False)

# First row of Otherinfo1 contains the original column names
original_cols = df.loc[0, "Otherinfo1"].split("\t")

# Remove the fake header row from ANNOVAR output
df = df[df["Chr"].astype(str) != "#Chr"].copy()

# Expand Otherinfo1 into separate columns
expanded = df["Otherinfo1"].astype(str).str.split("\t", expand=True)
expanded.columns = original_cols[:expanded.shape[1]]

expanded.head()

In [ ]:
#adding gnomad columns back

In [ ]:
gnomad_cols = [c for c in df.columns if c.startswith("gnomad41_")]

fixed_df = pd.concat(
    [
        expanded.reset_index(drop=True),
        df[gnomad_cols].reset_index(drop=True)
    ],
    axis=1
)

fixed_df.head()

In [ ]:
output_csv = Path("/project/knathans_shared/donetski/Run2_gnomad41_reannotation_with_header/fixed_combined_output/fixed_annovar_output_expanded_otherinfo.csv")
output_csv.parent.mkdir(parents=True, exist_ok=True)

fixed_df.to_csv(output_csv, index=False)

print("Saved:", output_csv)
print("Shape:", fixed_df.shape)